# A2 RNN's Visual encoder and Calculator.

# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using ***open-cv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split


from tensorflow.keras.layers import Dense, RNN, LSTM, Flatten, TimeDistributed, LSTMCell
from tensorflow.keras.layers import RepeatVector, Conv2D, SimpleRNN, GRU, Reshape, ConvLSTM2D, Conv2DTranspose

2026-01-06 12:07:06.867505: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-06 12:07:06.900046: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-06 12:07:15.817646: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [7]:
from scipy.ndimage import rotate
tf.random.set_seed(42)


# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)     # -||-

    for i in range(number_of_images): # Generate n different images
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)     # Draw lines with randomized coordinates
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA) # Draw lines with randomized coordinates

    return blank_images

def show_generated(images, n=5):
    plt.figure(figsize=(2, 2))
    for i in range(n**2):
        plt.subplot(n, n, i+1)
        plt.axis('off')
        plt.imshow(images[i])
    plt.show()

In [3]:
def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72' -> answer of the arithmetic text query
    y_img :  Stack of MNIST images corresponding to the answer (3 x 28 x 28)

    Images for digits are picked randomly from the whole MNIST dataset.
    """

    num_indices = [np.where(MNIST_labels==x) for x in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text, X_img, y_text, y_img = [], [], [], []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.

def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)


# Creating our data

The dataset consists of 20000 samples that (additions and subtractions between all 2-digit integers) and they have two kinds of inputs and label modalities:

  **X_text**: strings containing queries of length 5: ['  1+1  ', '11-18', ...]

  **X_image**: a stack of images representing a single query, dimensions: [5, 28, 28]

  **y_text**: strings containing answers of length 3: ['  2', '156']

  **y_image**: a stack of images that represents the answer to a query, dimensions: [3, 28, 28]

In [4]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters that are used in the queries (13 in total: digits 0-9, 2 operands [+, -], and a space character ' '.)
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (consists of two integers and an operand [e.g. '22+10'])
max_answer_length = 3    # Maximum length of the answer string (the longest resulting query string is ' 1-99'='-98')

# Create the data (might take around a minute)
(MNIST_data, MNIST_labels), _ = tf.keras.datasets.mnist.load_data()
X_text, X_img, y_text, y_img = create_data(highest_integer)
print(X_text.shape, X_img.shape, y_text.shape, y_img.shape)




(20000,) (20000, 5, 28, 28) (20000,) (20000, 3, 28, 28)


In [9]:
# Creating visual encoder training data, including ground-truth sequences used for teacher forcing.
size=0.1

X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_img, X_text_onehot, random_state=42, test_size=size
)

X_train_pt, X_val_pt, y_train_pt, y_val_pt = train_test_split(
    X_train_pt, y_train_pt, random_state=42, test_size=size/(1-size)
)

max_answer_length_pt = 6 # max anser length while using teacher forcing. pt = pretraining (old name)

y_train_in_pt = y_train_pt[:, :-1, :]
y_train_target_pt = y_train_pt[:, 1:, :]

y_val_in_pt = y_val_pt[:, :-1, :]
y_val_target_pt = y_val_pt[:, 1:, :]

y_test_in_pt = y_test_pt[:, :-1, :]
y_test_target_pt = y_test_pt[:, 1:, :]

In [10]:
# Calculator model data
size=0.1

X_train_calc, X_test_calc, y_train_calc, y_test_calc = train_test_split(
    X_text_onehot[:, :-1, :], y_text_onehot, random_state=42, test_size=size
)

X_train_calc, X_val_calc, y_train_calc, y_val_calc = train_test_split(
    X_train_calc, y_train_calc, random_state=42, test_size=size/(1-size)
)

max_answer_length_tf = 4 # max anser length while using teacher forcing

y_train_in_calc = y_train_calc[:, :-1, :]
y_train_target_calc = y_train_calc[:, 1:, :]

y_val_in_calc = y_val_calc[:, :-1, :]
y_val_target_calc = y_val_calc[:, 1:, :]

y_test_in_calc = y_test_calc[:, :-1, :]
y_test_target_calc = y_test_calc[:, 1:, :]

# My own helper functions

In the models below teacher forcing is used. For this the vocabulary will need a start and end token. Subsequently the one-hot encoding and decoding functions need to be altered to include these.

In [8]:
# Teacher forcing preparation
vocabulary_tf = list(unique_characters)+['<start>','<end>'] 

indices = {char:i for i, char in enumerate(vocabulary_tf)}
reverse_indices={i:char for i,char in enumerate(vocabulary_tf)}


# One-hot encoding
def encode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map=indices):
  n = len(labels)
  length = len(labels[0])+2 # for start of sequence <start> and end of sequence <end> tokens.

  one_hot = np.zeros([n, length, len(vocabulary)])
  for i, label in enumerate(labels):
    full_label = ['<start>'] + list(label) + ['<end>']
    m = np.zeros([length, len(vocabulary)])
    for j, char in enumerate(full_label):
       m[j, indices_map[char]] = 1
    one_hot[i] = m

  return one_hot

# One-hot decoding
def decode_labels_tf(labels, indices_map=reverse_indices):

    pred_indices = np.argmax(labels, axis=-1) 
    
    decoded_list = []
    for sequence in pred_indices:

        chars = [indices_map[i] for i in sequence if indices_map[i] not in ['<start>', '<end>', '<pad>']]
        decoded_list.append(''.join(chars))
        
    return decoded_list

X_text_onehot = encode_labels_tf(X_text)
y_text_onehot = encode_labels_tf(y_text)

print(X_text_onehot.shape, y_text_onehot.shape)

(20000, 7, 15) (20000, 5, 15)


# Visual encoder model

In [ ]:
# Masking layer for masked teacher forcing
def random_mask_layer(x, training=None, prob=0.5):

    def mask_logic():

        condition = tf.random.uniform([]) > prob
        return tf.cond(condition, lambda: x, lambda: tf.zeros_like(x))


    return tf.cond(tf.cast(training, tf.bool), 
                   mask_logic, 
                   lambda: tf.zeros_like(x))

In [ ]:
# Warmup training function for ease of use
def train_warmup(
        model,
        x,
        y,
        val_data,
        learning_rate=4.0e-4, # Initial LR
        weight_decay=1.0e-2  # Decoupled Weight Decay
):
    ## Compile
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=learning_rate,
        weight_decay=weight_decay,
    )

    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

    model.compile(
        loss=loss, 
        optimizer=optimizer, 
        metrics=['categorical_accuracy'])

    ## Training
    stopper_patience_warmup = 5

    early_stopper = tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        patience=stopper_patience_warmup,
        restore_best_weights=True
    )

    ### Warmup
    history = model.fit(x=x, y=y,
                epochs = 50,
                batch_size = 32,
                validation_data = val_data,
                callbacks = [early_stopper]) # only use an early stopper

    return history

# Fine-tune training function
def train_fine_tune(
        model,
        x,
        y,
        val_data,
        learning_rate=1.0e-5, # Initial LR
        weight_decay=5.0e-2,  # Decoupled Weight Decay
        scheduler_patience = 5,
        stopper_patience = 20
):
    # Fine tune training
    ## Recompile so AdamW moments are reset
    optimizer = tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

    model.compile(
        loss=loss, 
        optimizer=optimizer, 
        metrics=['categorical_accuracy'])

    early_stopper = tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        patience=stopper_patience,
        restore_best_weights=True
    )

    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=scheduler_patience,
        min_lr=1e-6,
        verbose=1
    )

    history = model.fit(x=x, y=y, 
                epochs = 60,
                batch_size = 32,
                validation_data = val_data,
                callbacks=[lr_scheduler, early_stopper],
                verbose=1)
    
    return history



### Building visual encoder

In [27]:
# Your code is: damn code

from tensorflow.keras.layers import Lambda, BatchNormalization, Activation, MaxPooling2D, LSTM, TimeDistributed, Dropout, Input, Add, LayerNormalization, Attention, Concatenate,GlobalAveragePooling2D
from tensorflow.keras.regularizers import L2, L1L2
from tensorflow.keras import layers, Sequential


# First we create a build-encoder function
def build_image2text_encoder(dropout, RLstrength, max_size=512): #old name which is an artifact of earlier ideas.
    
    # Initialize an encoder
    X_in = Input(shape = (5,28,28,1)) # 5 times a grayscale image

    data_augmentation = Sequential([
    layers.RandomRotation(0.05), # Rotate by ~18 degrees
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1), # Shift
    layers.RandomZoom(0.1), # Zoom in/out
    ], name="spatial_augmentation")

    augmentation_layer = layers.TimeDistributed(data_augmentation)(X_in)

    # Build encoder layers
    ## Block 1
    B1 = TimeDistributed(Conv2D(32, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(augmentation_layer)
    B1 = TimeDistributed(BatchNormalization())(B1)
    B1 = TimeDistributed(Activation('relu'))(B1)
    B1 = TimeDistributed(Dropout(dropout))(B1)
    B1_final = TimeDistributed(MaxPooling2D())(B1)


    ## Initialize the residual connection
    #residual = TimeDistributed(Activation('linear', name = "residual_branch"))(B1_final)
    residual2= TimeDistributed(Conv2D(64, (1,1), kernel_regularizer=L2(RLstrength/8), name = "residual_branch"))(B1_final)
    
    ## Block 2
    B2 = TimeDistributed(Conv2D(64, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(B1_final)
    B2 = TimeDistributed(BatchNormalization())(B2)
    B2 = TimeDistributed(Activation('relu'))(B2)
    B2_final = TimeDistributed(Dropout(dropout))(B2)

    ## Connect residual connection to output of two Conv2D blocks
    combined = Add()([B2_final, residual2])
    combined = TimeDistributed(Activation('relu'))(combined)

    ## Final pooling before the ConvLSTM2D layer
    final_pooling = TimeDistributed(MaxPooling2D(name='final_pooling'))(combined)


    ## Recurrent convolutional layers
    output, hidden, cell = ConvLSTM2D(
            filters=int(max_size/2), 
            kernel_size=(3,3), 
            padding='same',
            return_sequences=True, 
            use_bias=True, 
            return_state=True, 
            name='ConvLSTM', 
            dropout=dropout, #dropout
            #recurrent_dropout=dropout, #dropout
            kernel_regularizer = L2(RLstrength),
            recurrent_regularizer = L2(RLstrength))(final_pooling) #L2(RLstrength)

    encoder = tf.keras.Model(inputs=X_in, outputs=[output, hidden, cell], name = "encoder_model")
    return encoder

In [28]:
def build_image2text_pretraining(dropout = 0.5, max_size=512,RLstrength=1.0e-4, p = 0.5):

    vocab_size = 15

    X_in = Input(shape = (5,28,28,1), name = 'sequence')
    Y_in = Input(shape=(6, vocab_size))
    masked_ground_truth = Lambda(
        lambda x, training: random_mask_layer(x, training=training, prob=p),
        output_shape=(6, 15),
        name="random_masking_layer"
        )(Y_in)

    encoder = build_image2text_encoder(dropout,RLstrength, max_size=max_size)
    _, hidden, cell = encoder(X_in)
    
    h_flattened = GlobalAveragePooling2D(name='h_flattened')(hidden)#Flatten()(hidden)
    h_initial = Dense(max_size, kernel_regularizer=L2(RLstrength), name='h0')(h_flattened)

    c_flattened = GlobalAveragePooling2D(name='c_flattened')(cell)#Flatten(name='c_flattened')(cell)
    c_initial = Dense(max_size, kernel_regularizer=L2(RLstrength),name='c0')(c_flattened)

    ini_state = [h_initial, c_initial]
    
    output, hidden, cell = LSTM(
        max_size, 
        return_sequences = True,
        return_state=True, 
        dropout = dropout, 
        #recurrent_dropout = dropout, 
        name='lstm_pre_training',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        )(masked_ground_truth, initial_state = ini_state)
        
    dense = TimeDistributed(Dense(vocab_size, activation='softmax', name = 'decoder_dense_pre_training'))
    y_out = dense(output)

    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out, name = 'pretraining_model')
#    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
#    full.compile(
#        loss=loss, optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
#    )

    full.summary(expand_nested=True)
    return full



### Training visual encoder

In [ ]:
# Training the visual encoder. We train this beforehand.
## Building model
dropout=0.5
RLstrength=0
max_size=256 # Must be 128 at minimum. Must be an even number.
masking_prob = 0.7
image2text_pretraining = build_image2text_pretraining(dropout=dropout, RLstrength=RLstrength, max_size=max_size, p = masking_prob)

x=[X_train_pt, y_train_in_pt]
y=y_train_target_pt
val_data = ([X_val_pt, y_val_in_pt], y_val_target_pt)

learning_rate=4.0e-4 # Initial LR
weight_decay=1.0e-2  # Decoupled Weight Decay

train_warmup(image2text_pretraining,
             x = x,
             y = y,
             val_data = val_data,
             learning_rate=learning_rate,
             weight_decay=weight_decay)

"""
## Compile


optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

image2text_pretraining.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])


## Training
stopper_patience_warmup = 5

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience_warmup,
    restore_best_weights=True
)

### Warm-start
history = image2text_pretraining.fit(x=[X_train_pt, y_train_in_pt], y=y_train_target_pt, 
               epochs = 50,
               batch_size = 32,
               validation_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
               callbacks = [early_stopper])
"""


Model: "pretraining_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ sequence[0][0]    │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ input_layer_11 │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_42 │ 1)                │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        320 │ -                 │
│ time_distributed_43 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        128 │ -                 │
│ time_distributed_44 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_45 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_46 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_47 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │     18,496 │ -                 │
│ time_distributed_49 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │        256 │ -                 │
│ time_distributed_50 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_51 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_52 │ 64)               │            │                 

 Total params: 1,254,991 (4.79 MB)

 Trainable params: 1,254,799 (4.79 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/50


E0000 00:00:1767701703.851006 1754789 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_98/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_98/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_98/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/convolution_6' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_98/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_6', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_98/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_

499/500 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - categorical_accuracy: 0.3455 - loss: 2.0186

W0000 00:00:1767701726.141075 1754789 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/pretraining_model_1/random_masking_layer_1/cond/branch_executed/_88


500/500 ━━━━━━━━━━━━━━━━━━━━ 26s 45ms/step - categorical_accuracy: 0.4143 - loss: 1.8359 - val_categorical_accuracy: 0.4555 - val_loss: 1.7816
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.5886 - loss: 1.4709 - val_categorical_accuracy: 0.5111 - val_loss: 1.7039
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.7398 - loss: 1.1930 - val_categorical_accuracy: 0.6387 - val_loss: 1.3897
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.8549 - loss: 0.9768 - val_categorical_accuracy: 0.7902 - val_loss: 1.0926
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.9048 - loss: 0.8568 - val_categorical_accuracy: 0.7903 - val_loss: 1.0829
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.9312 - loss: 0.7871 - val_categorical_accuracy: 0.8774 - val_loss: 0.9007
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.9431 - 

In [30]:
# Fine tune training
## We choose a lower learning rate and higher weight decay.
learning_rate=1.0e-5
weight_decay=5.0e-2

## Recompile so AdamW moments are reset
optimizer = tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
image2text_pretraining.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])


scheduler_patience = 5
stopper_patience = 20

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

history = image2text_pretraining.fit(x=[X_train_pt, y_train_in_pt], y=y_train_target_pt, 
               epochs = 60,
               batch_size = 32,
               validation_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
               callbacks=[lr_scheduler, early_stopper],
               verbose=1)


Epoch 1/60


E0000 00:00:1767702577.417322 1754789 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_98/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_98/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_351/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/gradients/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Tanh_1_grad/TanhGrad' -> 'StatefulPartitionedCall/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_351/gra

500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - categorical_accuracy: 0.9887 - loss: 0.5916

W0000 00:00:1767702595.995824 1754789 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/pretraining_model_1/random_masking_layer_1/cond/branch_executed/_88


500/500 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - categorical_accuracy: 0.9895 - loss: 0.5890 - val_categorical_accuracy: 0.9703 - val_loss: 0.6387 - learning_rate: 1.0000e-05
Epoch 2/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - categorical_accuracy: 0.9898 - loss: 0.5869 - val_categorical_accuracy: 0.9692 - val_loss: 0.6415 - learning_rate: 1.0000e-05
Epoch 3/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - categorical_accuracy: 0.9904 - loss: 0.5859 - val_categorical_accuracy: 0.9702 - val_loss: 0.6391 - learning_rate: 1.0000e-05
Epoch 4/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 40ms/step - categorical_accuracy: 0.9904 - loss: 0.5854 - val_categorical_accuracy: 0.9684 - val_loss: 0.6417 - learning_rate: 1.0000e-05
Epoch 5/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - categorical_accuracy: 0.9909 - loss: 0.5842 - val_categorical_accuracy: 0.9683 - val_loss: 0.6423 - learning_rate: 1.0000e-05
Epoch 6/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - categorical_accuracy: 0.9906 - loss: 0.5853
Ep

In [57]:
# Export so that it can be used in A2_RNNs_Joep_full_model.ipynb without ruining the current optimal weights
image2text_pretraining.summary()
image2text_pretraining.save('visual_encoder.keras')

Model: "pretraining_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ sequence[0][0]    │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_25      │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h_flattened         │ (None, 128)       │          0 │ encoder_model[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c_flattened         │ (None, 128)       │          0 │ encoder_model[0]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_masking_lay… │ (None, 6, 15)     │          0 │ input_layer_25[0… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h0 (Dense)          │ (None, 256)       │     33,024 │ h_flattened[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ c0 (Dense)          │ (None, 256)       │     33,024 │ c_flattened[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_pre_training   │ [(None, 6, 256),  │    278,528 │ random_masking_l… │
│ (LSTM)              │ (None, 256),      │            │ h0[0][0],         │
│                     │ (None, 256)]      │            │ c0[0][0]          │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1… │ (None, 6, 15)     │      3,855 │ lstm_pre_trainin… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,254,991 (4.79 MB)

 Trainable params: 1,254,799 (4.79 MB)

 Non-trainable params: 192 (768.00 B)

# Calculator model

### Building calculator

In [50]:
# Calculator model
def build_text2text_calc(dropout = 0.5, max_size=512, max_answer_length_tf=4,RLstrength=1.0e-4, p = 0.5):

    vocab_size = len(vocabulary_tf)

    # Define input layer of full model
    X_in = Input(shape = (6, vocab_size), name = 'expression_input')
    Y_in = Input(shape=(max_answer_length_tf, len(vocabulary_tf)), name = "answer")
    masked_ground_truth = Lambda(
        lambda x, training: random_mask_layer(x, training=training, prob=p),
        output_shape=(4, 15),
        name="random_masking_layer"
        )(Y_in)

    # calculator encoder
    encoder_lstm = LSTM(max_size, return_state=True, return_sequences=True, name = 'calculator_encoder')
    key, hidden, cell = encoder_lstm(X_in)
    ini_state = [hidden, cell]    

    ## Calculator LSTM
    decoder_lstm = LSTM(
        max_size, 
        return_sequences = True, 
        return_state=True, 
        dropout = dropout, 
        recurrent_dropout = dropout, 
        name='decoder_lstm',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        )
    
    # Calculator decoder
    query, _, _ = decoder_lstm(masked_ground_truth, initial_state = ini_state)

    attention_block = Attention(name='attention_block')([query, key])

    combined = Concatenate(axis=-1, name = 'concat_q_A')([query, attention_block])
    combined = Dense(max_size, name = 'combined_dense', kernel_regularizer = L1L2(l1 = 5.0e-5, l2=RLstrength/2))(combined)

    residual = TimeDistributed(Dense(
        max_size, 
        use_bias=False, 
        kernel_regularizer = L2(RLstrength/2)
        ), 
        name = 'decoder_res_dense'
        )(Y_in)

    final = Add(name= 'decoder_with_residual')([combined, residual])
    final = TimeDistributed(Activation('relu'), name = "decoder_activation")(final)
    final = LayerNormalization(axis=-1, name='decoder_layer_norm')(final)
    
    y_out = TimeDistributed(Dense(vocab_size, activation='softmax'), name='decoder_dense')(final)


    # Full model step
    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out, name = 'calculator')
#    full.compile(
#        loss='categorical_crossentropy', optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
#    )

    full.summary(expand_nested=True)
    return full

### Training calculator

In [ ]:
# Training the calculator

## Building model
dropout=0.5
RLstrength=0
masking_prob = 0.5
text2text_calculator = build_text2text_calc(dropout=dropout, RLstrength=RLstrength, max_size=max_size, p = masking_prob)


learning_rate=5.0e-4 
weight_decay=1.0e-4
history_warmup = train_warmup(text2text_calculator, 
                              x=[X_train_calc, y_train_in_calc], 
                              y = y_train_target_calc, 
                              val_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
                              learning_rate=learning_rate,
                              weight_decay = weight_decay)

Model: "calculator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ answer (InputLayer) │ (None, 4, 15)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expression_input    │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_masking_lay… │ (None, 4, 15)     │          0 │ answer[0][0]      │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator_encoder  │ [(None, 6, 256),  │    278,528 │ expression_input… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 256),  │    278,528 │ random_masking_l… │
│                     │ (None, 256),      │            │ calculator_encod… │
│                     │ (None, 256)]      │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_block     │ (None, 4, 256)    │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_q_A          │ (None, 4, 512)    │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_block[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 4, 256)    │    131,328 │ concat_q_A[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_res_dense   │ (None, 4, 256)    │      3,840 │ answer[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_with_resid… │ (None, 4, 256)    │          0 │ combined_dense[0… │
│ (Add)               │                   │            │ decoder_res_dens… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_activation  │ (None, 4, 256)    │          0 │ decoder_with_res… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer_norm  │ (None, 4, 256)    │        512 │ decoder_activati… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 4, 15)     │      3,855 │ decoder_layer_no… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 696,591 (2.66 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50


W0000 00:00:1767703773.338249 1754789 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/calculator_1/random_masking_layer_1/cond/branch_executed/_59


500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.4826 - loss: 2.1916

W0000 00:00:1767703781.843450 1754789 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/calculator_1/random_masking_layer_1/cond/branch_executed/_31


500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - categorical_accuracy: 0.5346 - loss: 1.9851 - val_categorical_accuracy: 0.5562 - val_loss: 1.8124
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - categorical_accuracy: 0.5774 - loss: 1.7340 - val_categorical_accuracy: 0.6013 - val_loss: 1.6511
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - categorical_accuracy: 0.6167 - loss: 1.5826 - val_categorical_accuracy: 0.6428 - val_loss: 1.5023
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - categorical_accuracy: 0.6482 - loss: 1.4739 - val_categorical_accuracy: 0.6557 - val_loss: 1.4289
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - categorical_accuracy: 0.6715 - loss: 1.3986 - val_categorical_accuracy: 0.6734 - val_loss: 1.3719
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - categorical_accuracy: 0.6883 - loss: 1.3412 - val_categorical_accuracy: 0.7011 - val_loss: 1.3027
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - categorical_accuracy: 0.7053 - loss: 1

In [ ]:
"""
# Training the calculator

## Building model
dropout=0.5
RLstrength=0
text2text_calculator = build_text2text_calc(dropout=dropout, RLstrength=RLstrength, max_size=max_size)

## Compile
learning_rate=5.0e-4 
weight_decay=1.0e-4  

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

text2text_calculator.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])

## Training
stopper_patience_warmup = 5

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience_warmup,
    restore_best_weights=True
)

### Warm-start
history_warmup = text2text_calculator.fit(x=[X_train_calc, y_train_in_calc], y=y_train_target_calc, 
               epochs = 50,
               batch_size = 32,
               validation_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
               callbacks = [early_stopper])
"""

In [56]:
learning_rate=5.0e-4 
weight_decay=1.0e-5 

history_fine_tune = train_fine_tune(text2text_calculator,
                x=[X_train_calc, y_train_in_calc], 
                y=y_train_target_calc,
                val_data=([X_val_calc, y_val_in_calc], y_val_target_calc),
                learning_rate=learning_rate,
                weight_decay=weight_decay,
                scheduler_patience = 3,
                stopper_patience = 12)

Epoch 1/60


W0000 00:00:1767704344.440819 1754789 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/calculator_1/random_masking_layer_1/cond/branch_executed/_59


500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - categorical_accuracy: 0.9909 - loss: 0.6293

W0000 00:00:1767704352.719364 1754789 loop_optimizer.cc:934] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/calculator_1/random_masking_layer_1/cond/branch_executed/_31


500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - categorical_accuracy: 0.9910 - loss: 0.6295 - val_categorical_accuracy: 0.9921 - val_loss: 0.6266 - learning_rate: 5.0000e-04
Epoch 2/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - categorical_accuracy: 0.9912 - loss: 0.6280 - val_categorical_accuracy: 0.9938 - val_loss: 0.6195 - learning_rate: 5.0000e-04
Epoch 3/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - categorical_accuracy: 0.9915 - loss: 0.6259 - val_categorical_accuracy: 0.9936 - val_loss: 0.6177 - learning_rate: 5.0000e-04
Epoch 4/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - categorical_accuracy: 0.9937 - loss: 0.6199 - val_categorical_accuracy: 0.9944 - val_loss: 0.6141 - learning_rate: 5.0000e-04
Epoch 5/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - categorical_accuracy: 0.9927 - loss: 0.6214 - val_categorical_accuracy: 0.9925 - val_loss: 0.6180 - learning_rate: 5.0000e-04
Epoch 6/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - categorical_accuracy: 0.9936 - loss: 0.6185 - val_

In [ ]:
# Old
"""
### Recompile so AdamW momenta are reset
learning_rate=5.0e-4 
weight_decay=1.0e-5 

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

text2text_calculator.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])

scheduler_patience = 3
stopper_patience = 12

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

### Fine-tune training
history_calc = text2text_calculator.fit(x=[X_train_calc, y_train_in_calc], y=y_train_target_calc, 
               epochs = 60,
               batch_size = 32,
               validation_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
               callbacks=[lr_scheduler, early_stopper],
               verbose=1)
"""

In [ ]:
# Export so that it can be used in A2_RNNs_Joep_full_model.ipynb without ruining the current optimal weights
text2text_calculator.summary()
text2text_calculator.save('calculator.keras')